# learn-better — Google Colab setup

One-click setup for the [learn-better](https://github.com/dragosbo/learn-better) YouTube-to-study-material pipeline.

**How to use:** open this notebook in Colab (the *Open In Colab* badge in the repo README), then **Runtime → Run all**. Cells 1–2 install everything; cell 4 verifies. The cells are idempotent — safe to re-run after a disconnect.

> Colab runs Linux, so the Windows `.bat` runners don't apply here — call the tools with `python code/<script>.py` (optionally a `config/*.json` argument). All outputs land under `data/` (see `lib/paths.py`).

## Cell 1 — System deps + clone the repo (run once per session)

In [ ]:
import os

# ffmpeg is a SYSTEM binary (not pip-installable) that yt-dlp/faster-whisper need.
!apt-get install -y ffmpeg -q

REPO_URL = "https://github.com/dragosbo/learn-better.git"
REPO_DIR = "/content/learn-better"

# Idempotent: only clone if it isn't already here.
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

## Cell 2 — Python dependencies

In [ ]:
!pip install -q -r requirements.txt

## Cell 3 (optional) — Persist outputs to Google Drive

Colab sessions are ephemeral — anything under `data/` is lost when the runtime disconnects. Mount Drive and mirror the `data/` layout there if you want your audio / transcripts / summaries to survive. Skip this cell for a quick throwaway run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUT = "/content/drive/MyDrive/learn-better-output"
# Subdirs mirror lib/paths.py (all outputs live under data/).
for sub in ["audio", "audio_reencoded", "transcripts", "generated_transcripts",
            "summaries", "tts_output", "wordclouds", "whisper-model-cache"]:
    os.makedirs(f"{DRIVE_OUT}/{sub}", exist_ok=True)
print(f"Drive output folder ready: {DRIVE_OUT}")

## Cell 4 — Verify the environment

In [ ]:
# GPU check (optional): a T4 makes Whisper 5-10x faster.
# Runtime -> Change runtime type -> T4 GPU, then reconnect. faster-whisper uses CUDA automatically.
!nvidia-smi | head -3 || echo 'No GPU (CPU is fine; large Whisper models are slower).'

!ffmpeg -version | head -1
import yt_dlp, faster_whisper, pandas
print('✅ All imports OK — environment ready.')

## Cell 5 — Example run

The tools are config-driven. `transcribe_audio.py` takes an optional path to a `config/*.json`; with no argument it uses `config/config_transcribe.json`. Edit the config (or the source at the top of a script) to point at the playlist/channel/search/audio you want.

Common entry points (all write under `data/`):
- `python code/read_channel.py` — download audio + transcripts for a source
- `python code/transcribe_audio.py config/config_transcribe.json` — Whisper speech-to-text
- `python code/make_wordcloud.py config/config_wordcloud.json` — word-cloud JSON

In [ ]:
!python code/transcribe_audio.py config/config_transcribe.json